# Warehouse Analysis Notebook

This notebook loads the flattened Parquet tables from the Data Warehouse and displays them.

**Tables:** `genes`, `gene_seeds`, `powers`, `power_seeds`, `gene_regulation`, `gene_side_effects`.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
import pandas as pd

# Ensure we can import our modules
# sys.path.append(os.path.abspath("../super-services/src")) # Adjust if needed

from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe

In [ ]:
# Initialize Spark Session
bootstrap_spark_env()

spark = (SparkSession.builder
.appName("WarehouseViewer")
.config("spark.executor.memory", "4g")
.config("spark.driver.memory", "4g")
.getOrCreate())

In [ ]:
# Load Configuration
conf = utils.get_app_conf("generate_powers")
stage_root = conf.get_string("stage_root")
warehouse_root = os.path.join(stage_root, "warehouse")

print(f"Reading Warehouse from: {warehouse_root}")

## 1. Genes Table

In [ ]:
genes_df = spark.read.parquet(os.path.join(warehouse_root, "genes"))
display_scrollable_dataframe(genes_df.toPandas())

## 2. Powers Table

In [ ]:
powers_df = spark.read.parquet(os.path.join(warehouse_root, "powers"))
display_scrollable_dataframe(powers_df.toPandas())

## 3. Link Tables
**Gene Seeds, Power Seeds, Side Effects**

In [ ]:
gene_seeds_df = spark.read.parquet(os.path.join(warehouse_root, "gene_seeds"))
display_scrollable_dataframe(gene_seeds_df.limit(100).toPandas())

In [ ]:
power_seeds_df = spark.read.parquet(os.path.join(warehouse_root, "power_seeds"))
display_scrollable_dataframe(power_seeds_df.limit(100).toPandas())

In [ ]:
side_effects_df = spark.read.parquet(os.path.join(warehouse_root, "gene_side_effects"))
display_scrollable_dataframe(side_effects_df.limit(100).toPandas())

## 4. Gene Regulation (Graph)

In [ ]:
reg_df = spark.read.parquet(os.path.join(warehouse_root, "gene_regulation"))
display_scrollable_dataframe(reg_df.limit(100).toPandas())

## 5. Sanity Export (Excel)
Exporting top 10 rows of each table to `data/sanity/warehouse_top10.xlsx`.

In [ ]:
sanity_dir = os.path.join(stage_root, "sanity")
os.makedirs(sanity_dir, exist_ok=True)
out_path = os.path.join(sanity_dir, "warehouse_top10.xlsx")

# Forced Deletion to ensure freshness
if os.path.exists(out_path):
    try:
        os.remove(out_path)
        print(f"Deleted existing file: {out_path}")
    except OSError as e:
        print(f"Warning: Could not delete existing file {out_path}: {e}")

print(f"Exporting to {out_path}...")

with pd.ExcelWriter(out_path, mode='w') as writer:
    genes_df.limit(10).toPandas().to_excel(writer, sheet_name="Genes", index=False)
    powers_df.limit(10).toPandas().to_excel(writer, sheet_name="Powers", index=False)
    gene_seeds_df.limit(10).toPandas().to_excel(writer, sheet_name="Gene Seeds", index=False)
    power_seeds_df.limit(10).toPandas().to_excel(writer, sheet_name="Power Seeds", index=False)
    side_effects_df.limit(10).toPandas().to_excel(writer, sheet_name="Gene Side Effects", index=False)
    reg_df.limit(10).toPandas().to_excel(writer, sheet_name="Regulation", index=False)

print("Export Complete. File is fresh.")